# Imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['text.color'] = 'k'
mpl.rcParams['figure.figsize'] = (9,5)
plt.style.use('ggplot')

# Load Data

In [ ]:
data = pd.read_csv('data/raw/dataset_part1.csv', header = None)

In [ ]:
time_series = pd.read_csv("data/processed/time_series.csv", parse_dates=['month'], index_col="month")

In [ ]:
ts_decomp = pd.read_csv("data/processed/ts_decomposition.csv")

# Statiscal Properties

## Rolling Statistics

In [ ]:
rol_mean = time_series['active_users'].rolling(window=12).mean()
rol_std = time_series['active_users'].rolling(window=12).std()

In [ ]:
x = time_series.index.values
y = time_series.active_users.values

In [ ]:
plt.plot(x, y, color="blue", label="Original")
plt.plot(x, rol_mean, color="red", label="Rolling Mean")
plt.plot(x, rol_std, color='black', label="Rolling Std")
plt.legend()
plt.show()

## Autocorrelation

In [ ]:
plot_acf(x=time_series.active_users.values, ax=plt.gca(), lags=30)
plt.show()

In [ ]:
plot_pacf(x=time_series.active_users.values, ax=plt.gca(), lags=30, method='ywm')
plt.show()

## Dickey-Fuller Test

- H0: not stationary
- H1: stationary

In [ ]:
adf_test = np.array(adfuller(x=time_series.active_users.values, autolag='AIC'))

df_adf_test = pd.Series(data=np.r_[adf_test[[0,1,2,3]], [adf_test[4].get('5%')]],
                           index=['statistic', 'p-value', 'lags', 'used_observations', 'critical_value_5%'])
df_adf_test

Conclusion: no statiscal evidence to conclude that series is stationary.

# Test Function

In [ ]:
def test_stationarity(series, window=12):

    """
    Test stationarity of a time series.

    Arg:

    series: pandas.Series. The time series to be tested.

    window: the window time to calculate the rolling mean and rolling standard deviation.

    Returns:

    str: wether the series is stationary or not.

    """
    rol_mean = series.rolling(window=window).mean()
    rol_std = series.rolling(window=window).std()


    plt.plot(series, color="blue", label="Original")
    plt.plot(rol_mean, color="red", label="Rolling Mean")
    plt.plot(rol_std, color='black', label="Rolling Std")
    plt.legend()
    plt.show()

    plot_acf(x=series, ax=plt.gca(), lags=13)
    plt.show()


    adf_test = np.array(adfuller(x=series, autolag='AIC'))

    df_adf_test = pd.Series(data=np.r_[adf_test[[0,1,2,3]], [adf_test[4].get('5%')]],
                           index=['statistic', 'p-value', 'lags', 'used_observations', 'critical_value_5%'])

    print(df_adf_test)
    
    p_value = adf_test[1]

    if p_value > 0.05:
        print('\nConclusion:\nNo evidence to reject H0. The series is NOT stationary.')
    
    else:
        print('\nConclusion:\nReject H0. The series IS stationary.')

In [ ]:
test_stationarity(time_series['active_users'], window=12)